# Imports

In [1]:
import torch
import sys
import os
from torch_scatter import scatter
from torch import Tensor
sys.path.append('..')
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
from definitions import EXPERIMENTS_DIR, ROOT_DIR
from src.commands.train import Trainer, Config
from torch_geometric.data import HeteroData
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    hamming_loss,
)
from PIL import ImageColor

In [2]:
px.defaults.template = 'seaborn'

In [3]:
dir = os.path.join(ROOT_DIR, 'assets', 'articles_classification_results')

# Load the results

In [4]:
# TASK = 'hierarchical_classification'
EXPERIMENT = 'nantes-uni-2020_e5-v4'
CORPUS = 'nantes-uni-2020'
EXPERIMENT_DIR = os.path.join(EXPERIMENTS_DIR, EXPERIMENT)
RESULTS_DIR = os.path.join(EXPERIMENT_DIR, 'results')
HISTORIES_DIR = os.path.join(EXPERIMENT_DIR, 'histories')
CONFIG_PATH = os.path.join(EXPERIMENT_DIR, 'config.json')
SPLIT_PATH = os.path.join(EXPERIMENT_DIR, 'split_sub_dsis_4.pt')

In [5]:
history = pd.concat([
    pd.read_csv(os.path.join(HISTORIES_DIR, filename))
    for filename in os.listdir(HISTORIES_DIR)
])
history = history \
    .groupby(['epoch','phase','config','trial']) \
    .mean() \
    .reset_index()
history.head()

,epoch,phase,config,trial,subdomain_guidance_loss,lab_guidance_loss,loss,subdomain_guidance_acc,subdomain_guidance_f1,lab_guidance_acc,lab_guidance_f1,ssl
0,0,train,Conv_Attn_null_False,0,0.205094,0.153940,0.359033,0.919453,0.243549,0.941335,0.287482,NaN
1,0,train,Conv_Attn_null_False,1,0.196920,0.150571,0.347491,0.922982,0.249462,0.941475,0.287904,NaN
2,0,train,Conv_Attn_null_False,2,0.204393,0.150400,0.354793,0.919396,0.247016,0.941966,0.283047,NaN
3,0,train,Conv_Attn_null_False,3,0.201582,0.151033,0.352615,0.920625,0.241430,0.942130,0.286944,NaN
4,0,train,Conv_Attn_null_False_ssl,0,0.202915,0.154633,0.453214,0.920553,0.257154,0.939604,0.272128,0.95666


In [6]:
history_mean = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).mean() \
    .add_suffix('_mean')

history_std = history \
    .drop(columns = 'trial') \
    .groupby(['epoch','phase','config']).std() \
    .add_suffix('_std')

In [7]:
history = history_mean.join(other = history_std).reset_index()
history.head()

,epoch,phase,config,subdomain_guidance_loss_mean,lab_guidance_loss_mean,loss_mean,subdomain_guidance_acc_mean,subdomain_guidance_f1_mean,lab_guidance_acc_mean,lab_guidance_f1_mean,ssl_mean,subdomain_guidance_loss_std,lab_guidance_loss_std,loss_std,subdomain_guidance_acc_std,subdomain_guidance_f1_std,lab_guidance_acc_std,lab_guidance_f1_std,ssl_std
0,0,train,Conv_Attn_null_False,0.201997,0.151486,0.353483,0.920614,0.245364,0.941726,0.286345,NaN,0.003710,0.001657,0.004802,0.001677,0.003573,0.000382,0.002233,NaN
1,0,train,Conv_Attn_null_False_ssl,0.202258,0.151943,0.450135,0.920725,0.249356,0.940855,0.283759,0.959341,0.001793,0.002938,0.004955,0.000981,0.005201,0.001325,0.009711,0.010801
2,0,train,Conv_Max_null_False,0.206552,0.158254,0.364806,0.919971,0.233679,0.939067,0.263502,NaN,0.001750,0.000698,0.002176,0.000810,0.005501,0.000640,0.004737,NaN
3,0,train,Conv_Mean_null_False,0.203282,0.150755,0.354038,0.920557,0.237375,0.941802,0.288152,NaN,0.001496,0.002293,0.001403,0.001292,0.005976,0.001055,0.006755,NaN
4,0,train,Conv_Mean_null_False_ssl,0.209877,0.160472,0.733121,0.917142,0.229389,0.938032,0.268865,0.725544,0.001593,0.001423,0.010159,0.001556,0.009219,0.000949,0.003144,0.018814


# Load Data

In [8]:
config = Config(experiment=EXPERIMENT)
trainer = Trainer(experiment=config.experiment, config=config.training_config)
configs = trainer.get_configs()

In [9]:
data = trainer.load_data()
# data = trainer.filter_data(data)
# data = trainer.keep_largest_cc(data) if trainer.config.keep_largest_cc else data
data.split = trainer.create_or_load_split(data)
data['Article'].split = data.split

In [10]:
metadata = data.raw_metadata

# Learning Curves

In [11]:
def line_plot(
    df : pd.DataFrame,
    metric : str
) -> None:
    
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    exprs = df['config'].unique()

    for i, expr in enumerate(exprs):

        history_val = df.query(f"phase == 'val' and config == '{expr}'")
        color = colors[i % len(colors)]

        fig.add_trace(
            go.Scatter(
                name = expr,
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean'],
                mode='lines',
                line=dict(color=color),
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']+history_val[f'{metric}_std'],
                mode='lines',
                marker=dict(color="#444"),
                line=dict(width=0),
                showlegend=False
            )
        )

        fig.add_trace(
            go.Scatter(
                x=history_val['epoch'],
                y=history_val[f'{metric}_mean']-history_val[f'{metric}_std'],
                marker=dict(color="#444"),
                line=dict(width=0),
                mode='lines',
                fillcolor=f'rgb{ImageColor.getcolor(color, "RGB")}'.replace('rgb','rgba').replace(')',',0.2)'),
                fill='tonexty',
                showlegend=False
            )
        )

    fig.update_layout(
        xaxis_title='Epoch',
        yaxis_title=metric.capitalize(),
        width = 960,
        height = 800
    )

    fig.show()

In [12]:
line_plot(history, 'ssl')

In [13]:
line_plot(history, 'loss')

In [14]:
line_plot(history, 'subdomain_guidance_acc')

In [15]:
line_plot(history, 'subdomain_guidance_f1')

In [16]:
line_plot(history, 'lab_guidance_acc')

In [17]:
line_plot(history, 'lab_guidance_f1')

# Calculate other metrics

## Load Predictions

In [18]:
def load_results(results_dir : str) -> dict[tuple[str, str], Tensor]:

    results = dict()

    for filename in os.listdir(results_dir):

        parts = filename.split('.')[0].split('_')
        trial = parts[-1]
        expr = '_'.join(parts[1:-1])

        results[(expr,trial)] = torch.load(os.path.join(RESULTS_DIR, filename), map_location='cpu', weights_only=False)

    return results

In [19]:
results = load_results(RESULTS_DIR)

## Label Wise & Row Wise Metrics

In [20]:
def display_mean_std_df(df : pd.DataFrame) -> pd.DataFrame:

    columns = set([
        col.replace('_mean','')
        for col in df.columns if 'mean' in col
    ])

    new_df = pd.DataFrame()

    for col in df.columns:

        if not col.endswith('_mean') and not col.endswith('_std'):
            new_df[col] = df[col]

    for col in columns:
        new_df[col] = df[f'{col}_mean'].apply(lambda x : f'{x:.2%}') + ' ± ' + df[f'{col}_std'].apply(lambda x : f'{x:.2%}')

    return new_df

In [21]:
def get_experiment_params(expr : str) -> dict:

    parts = expr.split('_')
        
    return {
        "GNN Variant" : parts[0],
        "Readout Function" : parts[1],
        "Pooling Layer" : parts[2],
        "Undersampling" : parts[3]
    }

In [22]:
def test_articles_subset(
    data : HeteroData,
    min_p : float = 0.5
) -> Tensor:
    
    fold = 0
    split = data.split
    mask = split != fold

    entity_ids, articles_ids = data['Entity', 'APPEARIN', 'Article'].edge_index
    train_entities_ids = torch.unique(entity_ids[mask[articles_ids]])

    p = scatter(
        torch.isin(entity_ids, train_entities_ids).float(),
        articles_ids,
        dim=0,
        dim_size=data['Article'].x.size(0),
        reduce='mean'
    )

    p = p[~mask]

    node_ids = torch.arange(len(p)).to(p.device)

    return node_ids[p >= min_p]

In [23]:
def get_label_wise_metrics(
    label_name : str,
    graph : HeteroData,
    results : dict[tuple[str,str],Tensor],
    labels : list[str]
) -> tuple[pd.DataFrame,pd.DataFrame]:

    label_wise_metrics = []

    data = graph['Article']

    n_labels = data[f'y_{label_name}'] if label_name != 'lab' else data[f'y_{label_name}'][:,:-1]
    n_labels = n_labels.size(-1)

    for label in range(n_labels):
            
        for (expr,trial),result in results.items():

            y = data[f'y_{label_name}'][data.split == int(trial[-1])]

            y_hat = result[label_name][data.split == int(trial[-1])].sigmoid()
            y_hard = y_hat.gt(0.5).long()

            # subset_ids = test_articles_subset(graph, min_p=0.6)

            # y_hat = y_hat[subset_ids]
            # y_hard = y_hard[subset_ids]
            # y = y[subset_ids]
            
            label_wise_metrics.append({
                "team" : label,
                "expr" : expr,
                "trial" : trial,
                "accuracy" : accuracy_score(y[:,label], y_hard[:,label]),
                "f1_score" : f1_score(y[:,label], y_hard[:,label], zero_division=0),
                "precision" : precision_score(y[:,label], y_hard[:,label], zero_division=0),
                "recall" : recall_score(y[:,label], y_hard[:,label], zero_division=0),
                "auc" : roc_auc_score(y[:,label], y_hat[:,label])
            })

    label_wise_metrics = pd.DataFrame(label_wise_metrics)
    label_wise_metrics['team'] = label_wise_metrics['team'].apply(labels.__getitem__)
    label_wise_metrics['expr'] = label_wise_metrics['expr'].apply(str)

    label_wise_metrics_agg = label_wise_metrics.drop(columns="team").groupby(["expr","trial"]).mean().reset_index().sort_values(by = 'expr')

    label_wise_metrics_mean = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    label_wise_metrics_std = label_wise_metrics_agg \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    label_wise_metrics_agg = label_wise_metrics_mean.join(label_wise_metrics_std).reset_index()

    return label_wise_metrics_agg,label_wise_metrics

In [24]:
def get_row_wise_metrics(
    label : str,
    data : HeteroData,
    results : dict[tuple[str,str],Tensor],
) -> tuple[pd.DataFrame,pd.DataFrame]:

    row_wise_metrics = []

    for (expr,trial),result in results.items():

        y = data[f'y_{label}'][data.split == int(trial[-1])]
        y_hat = result[label][data.split == int(trial[-1])].sigmoid()
        y_hard = y_hat.gt(0.5).long()

        if label == 'lab':
            y = y[:,:-1]
            y_hat = y_hat[:,:-1]
            y_hard = y_hard[:,:-1]

        row_wise_metrics.append({
            "expr" : expr,
            "trial" : trial,
            "accuracy" : accuracy_score(y, y_hard),
            "f1_score" : f1_score(y, y_hard , average='macro', zero_division=0),
            "precision" : precision_score(y, y_hard , average='macro', zero_division=0),
            "recall" : recall_score(y, y_hard , average='macro', zero_division=0),
            "auc" : roc_auc_score(y, y_hat , average='macro'),
            "hamming_loss" : hamming_loss(y, y_hard)
        })

    row_wise_metrics = pd.DataFrame(row_wise_metrics)

    row_wise_metrics_mean = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .mean().add_suffix('_mean')

    row_wise_metrics_std = row_wise_metrics \
        .drop(columns="trial") \
        .groupby(by = ["expr"]) \
        .std().add_suffix('_std')

    row_wise_metrics_agg = row_wise_metrics_mean.join(row_wise_metrics_std).reset_index()

    return row_wise_metrics_agg,row_wise_metrics

In [25]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('subdomain', data, results, metadata['labels']['subdomain'])
display_mean_std_df(label_wise_metrics_agg)

,expr,accuracy,precision,f1_score,recall,auc
0,Conv_Attn_null_False,93.85% ± 0.33%,72.59% ± 2.10%,61.67% ± 1.47%,55.01% ± 1.72%,93.39% ± 0.48%
1,Conv_Attn_null_False_ssl,93.97% ± 0.28%,72.26% ± 2.55%,62.86% ± 1.14%,57.35% ± 1.58%,93.91% ± 0.41%
2,Conv_Max_null_False,94.48% ± 0.17%,77.07% ± 1.59%,64.71% ± 1.76%,57.27% ± 2.33%,94.90% ± 0.18%
3,Conv_Mean_null_False,94.20% ± 0.18%,72.75% ± 1.89%,63.18% ± 0.74%,57.07% ± 0.95%,94.38% ± 0.30%
4,Conv_Mean_null_False_ssl,94.25% ± 0.21%,71.66% ± 2.78%,62.61% ± 0.98%,56.77% ± 1.02%,94.46% ± 0.32%


In [26]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('subdomain', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,hamming_loss,accuracy,precision,f1_score,recall,auc
0,Conv_Attn_null_False,6.15% ± 0.33%,40.17% ± 2.04%,72.59% ± 2.10%,61.67% ± 1.47%,55.01% ± 1.72%,93.39% ± 0.48%
1,Conv_Attn_null_False_ssl,6.03% ± 0.28%,40.40% ± 0.57%,72.26% ± 2.55%,62.86% ± 1.14%,57.35% ± 1.58%,93.91% ± 0.41%
2,Conv_Max_null_False,5.52% ± 0.17%,43.24% ± 1.37%,77.07% ± 1.59%,64.71% ± 1.76%,57.27% ± 2.33%,94.90% ± 0.18%
3,Conv_Mean_null_False,5.80% ± 0.18%,41.94% ± 0.79%,72.75% ± 1.89%,63.18% ± 0.74%,57.07% ± 0.95%,94.38% ± 0.30%
4,Conv_Mean_null_False_ssl,5.75% ± 0.21%,41.82% ± 1.35%,71.66% ± 2.78%,62.61% ± 0.98%,56.77% ± 1.02%,94.46% ± 0.32%


In [27]:
label_wise_metrics_agg,label_wise_metrics = get_label_wise_metrics('lab', data, results, metadata['labels']['lab'])
display_mean_std_df(label_wise_metrics_agg)

,expr,accuracy,precision,f1_score,recall,auc
0,Conv_Attn_null_False,95.84% ± 0.37%,78.33% ± 2.01%,68.37% ± 4.97%,62.61% ± 4.83%,96.20% ± 0.70%
1,Conv_Attn_null_False_ssl,95.95% ± 0.26%,78.31% ± 3.00%,69.47% ± 3.53%,64.46% ± 3.72%,96.39% ± 0.27%
2,Conv_Max_null_False,96.07% ± 0.45%,81.70% ± 4.26%,68.43% ± 4.20%,61.59% ± 3.65%,96.83% ± 0.56%
3,Conv_Mean_null_False,96.08% ± 0.43%,80.05% ± 4.97%,71.65% ± 4.46%,66.50% ± 4.56%,96.72% ± 0.32%
4,Conv_Mean_null_False_ssl,96.04% ± 0.43%,78.21% ± 4.14%,70.81% ± 4.31%,66.43% ± 4.47%,96.59% ± 0.33%


In [28]:
row_wise_metrics_agg,row_wise_metrics = get_row_wise_metrics('lab', data['Article'], results)
metrics = display_mean_std_df(row_wise_metrics_agg)
metrics

,expr,hamming_loss,accuracy,precision,f1_score,recall,auc
0,Conv_Attn_null_False,4.16% ± 0.37%,71.69% ± 2.21%,78.33% ± 2.01%,68.37% ± 4.97%,62.61% ± 4.83%,96.20% ± 0.70%
1,Conv_Attn_null_False_ssl,4.05% ± 0.26%,72.63% ± 1.66%,78.31% ± 3.00%,69.47% ± 3.53%,64.46% ± 3.72%,96.39% ± 0.27%
2,Conv_Max_null_False,3.93% ± 0.45%,72.19% ± 2.03%,81.70% ± 4.26%,68.43% ± 4.20%,61.59% ± 3.65%,96.83% ± 0.56%
3,Conv_Mean_null_False,3.92% ± 0.43%,72.86% ± 3.06%,80.05% ± 4.97%,71.65% ± 4.46%,66.50% ± 4.56%,96.72% ± 0.32%
4,Conv_Mean_null_False_ssl,3.96% ± 0.43%,72.32% ± 3.08%,78.21% ± 4.14%,70.81% ± 4.31%,66.43% ± 4.47%,96.59% ± 0.33%


- Visualizations

In [28]:
def bar_plot(df : pd.DataFrame, metric : str) -> None:

    fig = px.bar(
        data_frame = df.sort_values(by = f'{metric}_mean'),
        x = 'expr',
        y = f'{metric}_mean',
        barmode = 'group',
        error_y = f'{metric}_std',
        text = f'{metric}_mean',
        width = 950,
        height = 800
    )

    fig.update_layout(
        xaxis_title = 'Experiment',
        yaxis_title = metric.capitalize().replace('_',' '),
    )

    fig.update_traces(
        textposition = 'outside',
        texttemplate = '%{text:.2%}'
    )

    fig.show()

In [29]:
bar_plot(label_wise_metrics_agg, 'accuracy')

In [30]:
bar_plot(label_wise_metrics_agg, 'f1_score')

In [31]:
bar_plot(row_wise_metrics_agg, 'accuracy')